# VecDB Bulk Loading – Integrated Embeddings

## 1. Scenario Overview
This focused walkthrough provisions a model-backed VecDB table for support-ticket style metadata. Use it when you only have JSON/text fields (no pre-computed vectors) and want the database to run embeddings on your behalf.

Upload `bulk_loading/data/bulktable_integrated_embedding.csv` to Object Storage and store the signed URL in `INTEGRATED_CSV_URL` before running the cells below.


## Architecture at a Glance

```text
Integrated Embedding Bulk Load

  local fixture                         external object                    VecDB runtime
+-------------------+                 +------------------+              +-----------------------+
| METADATA + ID CSV | -- upload/PAR ->| Object Storage   | -- read ---> | bulk load operation   |
| no vector column  |                 | HTTPS CSV object |              | table=SUPPORT_AUTO... |
+---------+---------+                 +------------------+              +-----------+-----------+
          |                                                                        |
          | local preview                                                          | uses embed_params
          v                                                                        v
+-------------------+                                                +---------------------------+
| verify payload    |                                                | existing EMBED_MODEL      |
| row count, IDs,   |                                                | generates dense vectors   |
| metadata shape    |                                                +-------------+-------------+
+-------------------+                                                              |
                                                                                   v
                                                                      +---------------------------+
                                                                      | stored rows                |
                                                                      | ID + METADATA + VECTOR     |
                                                                      +-------------+-------------+
                                                                                    |
                                                                                    v
                                                                      +---------------------------+
                                                                      | list_vectors + cleanup     |
                                                                      +---------------------------+
```

This notebook is the only bulk-loading scenario where the CSV does not provide vectors. The important dependency is the existing VecDB embedding model named by `EMBED_MODEL`.


## What to Validate

- The local CSV preview should show `METADATA` and `ID`, with no vector column.
- The configured `EMBED_MODEL` must already exist in VecDB before the load job runs.
- After the load job succeeds, `list_vectors` should return the CSV IDs, metadata, and VecDB-generated vectors.


## 2. Setup





### Install Required Packages
Run the following cell once per environment to install / upgrade `oracle-vecdb`, `python-dotenv`, and `pandas`. Skip this step if you already have the dependencies available in your kernel.



In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas



### Load Credentials and Configure Demo Inputs
This cell reads `.env` for `VECDB_REST_URL`, `VECDB_USERNAME`/`VECDB_PASSWORD`, wires up the published Object Storage URLs, and defines table names plus the embedding model used later. Update the `.env` file or override any env var before running.



In [ ]:
import os
from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration

print('Loading VecDB environment variables...')
load_dotenv()
from dotenv import find_dotenv
print(find_dotenv())

resolved_host = os.getenv('VECDB_REST_URL')
resolved_user = os.getenv('VECDB_USERNAME')
resolved_password = os.getenv('VECDB_PASSWORD')
resolved_access_token = os.getenv('VECDB_ACCESS_TOKEN')
print(f'Resolved REST endpoint: {resolved_host}')
print(f'Resolved user: {resolved_user}')


config_kwargs = {"rest_url": resolved_host}
if resolved_access_token:
    config_kwargs["access_token"] = resolved_access_token
else:
    config_kwargs["username"] = resolved_user
    config_kwargs["password"] = resolved_password
config = Configuration(**config_kwargs)
if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('Disabled SSL verification (self-signed certificates).')

vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print('Connected to', config.rest_url)
print('Auth method:', auth_method)
print('VecDB client ready for this workflow.')

INTEGRATED_CSV_URL = os.getenv('INTEGRATED_CSV_URL')
if not INTEGRATED_CSV_URL:
    raise RuntimeError('Set INTEGRATED_CSV_URL with the Object Storage URL for this scenario.')
print('Integrated CSV URL:', INTEGRATED_CSV_URL)

EMBED_MODEL = os.getenv('EMBED_MODEL')
INTEGRATED_TABLE = os.getenv('INTEGRATED_TABLE')
print('Integrated table:', INTEGRATED_TABLE)
print('Embedding model:', EMBED_MODEL)


### Local CSV Checkpoint

This checkpoint reads the CSV fixture from `bulk_loading/data/` only. It is not a VecDB response. Run the cell to preview the exact local file that should later be uploaded to Object Storage for the bulk load job.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display


def _resolve_fixture(filename):
    for candidate in (Path('data') / filename, Path('bulk_loading') / 'data' / filename):
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not find {filename}. Run this notebook from bulk_loading/ or notebooks/vecdb/.')

fixture_path = _resolve_fixture('bulktable_integrated_embedding.csv')
fixture_df = pd.read_csv(fixture_path)
metadata_preview = fixture_df['METADATA'].apply(json.loads).apply(pd.Series)
preview_df = pd.concat([fixture_df[['ID']], metadata_preview], axis=1)

scenario_summary = pd.DataFrame([
    ('Scenario', 'Integrated embeddings'),
    ('Local fixture', str(fixture_path)),
    ('Rows in fixture', len(fixture_df)),
    ('CSV columns', ', '.join(fixture_df.columns)),
    ('ID mode', 'IDs provided by CSV'),
    ('Vector source', 'Generated by VecDB using EMBED_MODEL'),
    ('Target table', INTEGRATED_TABLE),
    ('Object Storage env var', 'INTEGRATED_CSV_URL'),
], columns=['Setting', 'Value'])

display(scenario_summary)
display(preview_df.head())


## 3. Bulk load into a table with integrated embedding
Provision a fresh support-ticket table using `create_vector_table`. The table generates embeddings inside VecDB by pointing `embed_metadata_jsonpath` to the `DESCRIPTION` field in each CSV row.


### Confirm VecDB embedding model
Ensure `EMBED_MODEL` already exists in your VecDB environment (deploy it ahead of time) and is set in `.env`. This notebook only references that model; it does not upload or load ONNX artifacts.

In [ ]:
if not EMBED_MODEL:
    raise RuntimeError('Set EMBED_MODEL to an existing VecDB embedding model name.')
print('Using VecDB embedding model:', EMBED_MODEL)

In [ ]:
print('Resetting integrated-embedding demo table...')
try:
    vecdb.drop_vector_table(name=INTEGRATED_TABLE)
    print(f'Dropped existing table {INTEGRATED_TABLE}')
except Exception:
    print(f'Table {INTEGRATED_TABLE} not present; creating new table')

vecdb.create_vector_table(
    name=INTEGRATED_TABLE,
    comment='Support tickets demo table with integrated embeddings',
    annotations={'DATASET': 'support_tickets'},
    table_params={"auto_generate_id":False},
    embed_params={
        'model': EMBED_MODEL,
        'embed_metadata_jsonpath': 'DESCRIPTION',
    },
)
print('Ready for bulk load:', INTEGRATED_TABLE)



## 4. Bulk Load Metadata (Integrated Embeddings)
Call `load_vectors` with the signed URL for `bulktable_integrated_embedding.csv`. Because the CSV only contains metadata + IDs, VecDB handles embedding creation during the load job.


In [ ]:
print('Submitting integrated embeddings bulk load job...')
load_job = vecdb.load_vectors(
    table_name=INTEGRATED_TABLE,
    url=INTEGRATED_CSV_URL,
)
EMBED_JOB_NAME = getattr(load_job, 'job_name', None) 
if EMBED_JOB_NAME:
    print('Load job name:', EMBED_JOB_NAME)
else:
    print('Raw load job response:', load_job)



## 5. Inspect Load Job Output
Use `list_vector_load_jobs`, `describe_vector_load_job`, and `get_vector_load_job_log` to monitor the ingest. When a job name is available the cell prints end-to-end metadata.



In [ ]:
jobs = vecdb.list_vector_load_jobs()
print('Vector load jobs:', jobs)

job_name = globals().get('EMBED_JOB_NAME')
if job_name:
    details = vecdb.describe_vector_load_job(load_job_name=job_name)
    print('Integrated job details:', details)
    try:
        logs = vecdb.get_vector_load_job_log(load_job_name=job_name)
        log_preview = str(logs)
        print('Log preview:', log_preview[:500])
    except Exception as exc:
        print('Unable to fetch log output:', exc)
else:
    print('Run the load job cell to capture a job identifier for inspection.')



## 6. List Vectors
Once the integrated job succeeds, page through the table with `list_vectors`. Expect IDs and metadata coming from the CSV plus embeddings created on the fly.



In [ ]:
vectors = vecdb.list_vectors(table_name=INTEGRATED_TABLE, limit=5)
print('Listed vectors count:', len(vectors.items))
for item in vectors.items:
    print(item.id, item.metadata, item.dense_vector)

## Cleanup
Drop the demo table so repeated runs stay isolated.

In [ ]:
print('Dropping integrated embeddings table...')
try:
    vecdb.drop_vector_table(name=INTEGRATED_TABLE)
    print('Dropped', INTEGRATED_TABLE)
except Exception as exc:
    print('Unable to drop', INTEGRATED_TABLE, exc)